In [0]:
from pyspark.sql.session import SparkSession
print(spark)
spark1=SparkSession.builder.getOrCreate()
print(spark1)

In [0]:
%sql
create catalog if not exists project_1;
create database if not exists project_1.db_1;
create volume if not exists project_1.db_1.vol1;

In [0]:
dbutils.fs.mkdirs("/Volumes/project_1/db_1/vol1/projdir_1")

In [0]:
dbutils.fs.ls("/Volumes/project_1/db_1/vol1")

## If no options are used in CSV function, default functionality is
#### 1. ',' as delimiter
#### 2. c0,c1,c2 as column headers
#### 3. treat all columns as string

###  inferSchema needs to used cautiously because it does immediate evaluation and execution that is ok for small datasets, but not ideal for large datasets or predefined schema dataset. It literally kills performance

In [0]:
df1_csv = spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs")
display(df1_csv.limit(10))
df1_csv.printSchema()

#### Different methods of create dataframe with CSV and other module

In [0]:
df2_csv = spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs", header=False, inferSchema=True) #Method 1
display(df2_csv.limit(10))
df2_csv = spark.read.options(header=False, inferSchema=True).format("csv").load("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs") #Method 2
display(df2_csv.limit(10))
df2_csv = spark.read.option("header","False").option("inferSchema","True").format("csv").load("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs") #Method 3
display(df2_csv.limit(10))


In [0]:
df2_csv = spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs", header=False, inferSchema=True).toDF("ID", "FIRST_NAME","LAST_NAME","AGE","OCCUPATION")
display(df2_csv.limit(10))
df3_csv = spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs", header=True, inferSchema=True)# if there is no header we are suppose to give header=False always otherwise it considers first row as headers, we might end up losing data
display(df3_csv.limit(10))
df3_csv.printSchema()

#### As a default it considers only comma as a delimiter, but here ~ is delimiter, it combines all columns and rows as on. 
#### In order to avoid that we use (sep='~') to seperate columns using spark rather than using complex split in SQL

In [0]:
df4_csv=spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs_1.txt",header=True, sep="~", inferSchema=True,samplingRatio=0.10) 
df4_csv.show()

### Reading data from multiple files

In [0]:
# custs* represents all filename matching custs and read all data together
df4_csv=spark.read.csv("dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs*",header=True, sep="~", inferSchema=True) 
#df4_csv.show()
print(df4_csv.count())

### Reading multiple location files together

In [0]:
df5_csv=(
    spark.read.csv(path=["dbfs:///Volumes/project_1/db_1/vol1/projdir_1/custs_1.txt","dbfs:///Volumes/catalog_one/schema_one/volume_one/directory_one/patients.csv"],header=True, sep="~", inferSchema=True))
print(df5_csv.count())


In [0]:
df_multiple_source=(
    spark.read.format("csv")
    .option("header", "True")
    .option("inferSchema", "True")
    .option("sep",",")
    .option("pathGlobFilter","custs_header*")
    .option("recursiveFileLookup","True")
    .load(path=["dbfs:///Volumes/catalog_one/schema_one/volume_one/Source/TX", "dbfs:///Volumes/catalog_one/schema_one/volume_one/Source/NY"]))
print(df_multiple_source.count())
#df_multiple_source.printSchema()
#df_multiple_source.show(10)


In [0]:
df2=spark.read.csv(path=["dbfs:///Volumes/catalog_one/schema_one/volume_one/Source/NY/","dbfs:///Volumes/catalog_one/schema_one/volume_one/Source/TX/"], header=True,inferSchema=True, sep=",", pathGlobFilter="custs_header*", recursiveFileLookup=True)
print(df2.count())

In [0]:
from pyspark.sql.functions import count, col

df2.groupBy("_metadata.file_path").count().orderBy("count").show(truncate=False)


In [0]:
#struct_type=("id integer, first_name string, last_name string, Age int, prof string")
df4= spark.read.schema(struct_type).csv("/Volumes/catalog_one/schema_one/volume_one/Source/NY/HM/custs_header_1",header=True)
print(df4.printSchema())
df4.show()

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
custom_schema=StructType([StructField("id",IntegerType(),True),StructField("fname",StringType(),True),StructField("lname",StringType(),True),StructField("Age",IntegerType(),True),StructField("prof",StringType())])
df4=spark.read.schema(custom_schema).csv("/Volumes/catalog_one/schema_one/volume_one/Source/NY/HM/DB_sample.txt",sep=",",header=False)
display(df4)
                                                                                                                  
            

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
s1=StructType([StructField("id", IntegerType(), False),StructField("first_name", StringType(),False),StructField("last_name", StringType(),False), StructField("Age", IntegerType(), True),StructField("occupation",StringType(), True)])

In [0]:
df_xml=spark.read.xml("/Volumes/catalog_one/schema_one/volume_one/directory_one/12mb (1).xml",rowTag="book")
df_xml.show(10)